# Basic System of 2 Linear PDEs via Native Complex Formulation
## The 2D Linear Schrödinger Equation

This notebook demonstrates **Approach 1: Native Complex** using a very basic system of linear PDEs: the 2D free-particle Schrödinger equation. 

Instead of writing a coupled system of two real equations, we define a single complex-valued field $\psi(x, y, t) = u + iv$. 

---

## 1. The Governing Equation & Equivalence

**Complex Form (1 PDE):**
$$
\partial_t \psi = i \nabla^2 \psi
$$

**Equivalent Real System (2 Linear PDEs):**
By substituting $\psi = u + iv$ and separating real and imaginary parts, we get:
$$
\begin{align}
\partial_t u &= -\nabla^2 v \\
\partial_t v &= \phantom{-}\nabla^2 u
\end{align}
$$
*This shows that the real part $u$ evolves via the negative Laplacian of $v$, and the imaginary part $v$ evolves via the Laplacian of $u$. Writing it natively as one complex PDE is much cleaner.*

---

## 2. Reformulation for the Solver

We expand the linear operator in Fourier space, where $\nabla^2 \to -k^2 = -(\xi^2 + \eta^2)$:

$$
\text{Linear symbol:} \quad -i(\xi^2 + \eta^2)
$$

The equation in the solver's format:

$$
\partial_t \psi = \text{psiOp}\left(-i(\xi^2 + \eta^2), \psi\right)
$$

# Implementation
## 0. Imports

In [ ]:
from solver import PDESolver, psiOp
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters

In [ ]:
# ── Grid and Time ──
Lx, Ly = 20.0, 20.0   # Domain size
Nx, Ny = 128, 128     # Resolution

Lt, Nt = 10.0, 100   # Simulation time and steps
n_frames = 100        # Frames for animation

## 2. Grid setup

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')

## 3. SymPy symbols and principal symbol

In [ ]:
t, x, y = sp.symbols('t x y', real=True)
# Using 'xi' and 'eta' for Fourier wavenumbers
xi, eta = sp.symbols('xi eta', real=True)

# Define psi as a complex-valued field
psi_field = sp.Function('psi')(t, x, y)

# ── Linear symbol in Fourier space ──
# From: ∂ψ/∂t = i ∇²ψ
# Fourier: ∇² → -(ξ² + η²)
# So: -i(ξ² + η²)

k2 = xi**2 + eta**2
symbol_linear = -sp.I * k2

print('Principal symbol (linear part):')
print('  a(ξ, η) = ', symbol_linear)

## 4. The Linear Schrödinger equation

In [ ]:
# ∂ψ/∂t = psiOp(-i(ξ² + η²), ψ)

equation = sp.Eq(
    sp.diff(psi_field, t),
    psiOp(symbol_linear, psi_field)
)

print('2D Linear Schrödinger Equation:')
print('  ∂ψ/∂t = psiOp(-i(ξ² + η²), ψ)')
print('\n(Equivalent to the 2 linear PDEs: ∂u/∂t = -∇²v  and  ∂v/∂t = ∇²u)')

## 5. Initial conditions

In [ ]:
def initial_condition_psi(xx, yy):
    """
    A Gaussian wave packet with an initial phase gradient.
    The phase gradient gives the packet an initial 'momentum',
    causing it to move diagonally while dispersing.
    """
    width = 1.5
    k0 = 4.0  # Initial wavenumber (momentum)
    
    gaussian = np.exp(-(xx**2 + yy**2) / width**2)
    phase = np.exp(1j * k0 * (xx + yy) / np.sqrt(2))
    
    return gaussian * phase

print("Using: Gaussian wave packet with initial momentum")

## 6. Solver setup

In [ ]:
solver = PDESolver(equation)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='periodic',
    initial_condition=initial_condition_psi,
    n_frames=n_frames,
    plot=True,
)

## 7. Solve

In [ ]:
frames = solver.solve()

## 8. Visualization

In [ ]:
plt.rcParams['animation.embed_limit'] = 2**128

# Visualize the magnitude (probability density |ψ|² or amplitude |ψ|)
ani = solver.animate(
    component='abs',      # 'abs' shows the amplitude |ψ|
    overlay=None,
    mode='surface',       # 'surface' shows the wave packet clearly
    physical=True
)

HTML(ani.to_jshtml())

In [ ]:
ani.save('linear_schrodinger_complex.mp4', writer='ffmpeg', fps=20, dpi=100)
print("✅ Saved to linear_schrodinger_complex.mp4")

print("\n" + "="*60)
print("OBSERVATIONS:")
print("="*60)
print("1. The wave packet moves diagonally due to the initial phase.")
print("2. It continuously spreads out (dispersion) because different")
print("   Fourier modes travel at different phase velocities.")
print("3. If your solver throws a complex error, switch to the")
print("   2-equation real split (Approach 2).")
print("="*60)